# Make subsets

use multilabel stratifier, but we'll assign selected samples to be in the last subfold using the quality indicator, thus ensuring that all subsets will have at least 1 of each label

In [ ]:
import os

import numpy as np
import pandas as pd

from collections import defaultdict
from pass_pclr.defines import CINC_TARGETS

from _multilabel_stratified_sampling import stratify

dataset_path = "/opt/gpudata/ecg/cinc-2020"
subset_root = "/opt/gpudata/ecg/temp"

In [ ]:
df = pd.read_csv(os.path.join(dataset_path, "georgia.csv"))
train_df = df[df["split"] == "train"]
val_test_df = df[df["split"].isin(["val", "test"])]

In [ ]:
assert (df["age"] < 100).all()
binned_age = pd.cut(train_df["age"], [0, 20, 40, 60, 80, 100])
one_hot_age = pd.get_dummies(binned_age).astype(int)
one_hot_sex = pd.get_dummies(train_df["sex"]).astype(int)
labels = train_df[CINC_TARGETS]
stratifier = pd.concat([one_hot_sex, one_hot_age, labels], axis=1)

In [ ]:
# no patient identifiers in this dataset, assume each ECG belongs to a unique patient
n_samples, n_classes = stratifier.shape

# stratify creates a patient ID based on the passed in list
# since we use a subset of the entire dataset, we need a way
# to map back to the IDs of the original whole dataset
remap = {i: v for i, v in enumerate(train_df.index)}

# to ensure the train set always has every possible label, we hijack the stratifier's
# notion of quality to ensure that the final split has those patients/ecgs
rng = np.random.default_rng(seed=42)
selected_idxs = set()
for target in CINC_TARGETS:
    # this gets us indices in the subset, not the original dataset patient IDs
    candidates = np.argwhere(labels[target]).flatten()
    idx = rng.choice(candidates)
    selected_idxs.add(idx)
assert len(selected_idxs) < 256 # should not be larger than smallest subset we aim to make
qualities = [4 if i in selected_idxs else 2 for i in range(n_samples)]

label_lists = [np.where(row)[0].tolist() for row in stratifier.to_numpy()]

stratified_ids, stratified_labels = stratify(
    data=label_lists,
    classes=list(range(n_classes)),
    ratios=[0.03125] * 32, # need to construct powers of 2 from 8192 to 256
    qualities=qualities,
    ecgs_per_patient=[1] * n_samples,
    nr_clean_folds=1,
    random_seed=2, # find a random seed that makes the last subfold precisely size 256
)

In [ ]:
# sort the subfolds by size for easier manual inspection
subset_idxs_by_size = defaultdict(list)
for i, subset_ids in enumerate(stratified_ids):
    size = len(subset_ids)
    subset_idxs_by_size[size].append(i)

In [ ]:
# select folds to use to construct target, prefer 256 sized subsets
# last fold will has the preselected samples to ensure all labels accounted for
# and is therefore present in all constructed subsets
{k: subset_idxs_by_size[k] for k in sorted(subset_idxs_by_size.keys())}

In [ ]:
def make_train_subset(subfolds: list[int], name: str) -> pd.DataFrame:
    # remap idxs back to the original train set
    train_idxs = [remap[x] for subfold in subfolds for x in stratified_ids[subfold]]
    subset_df = pd.concat([train_df.loc[train_idxs], val_test_df])

    subset_path = os.path.join(subset_root, f"cinc-2020-{name}")
    os.makedirs(subset_path, exist_ok=True)

    subset_df.to_csv(os.path.join(subset_path, "georgia.csv"), index=False)

    # also link source waveform data
    os.symlink(
        src=os.path.join(dataset_path, "training"),
        dst=os.path.join(subset_path, "training"),
    )

    return subset_df

In [ ]:
for name, target_size, folds in [
    ("4k", 4096, [3, 6, 8, 12, 13, 15, 16, 18, 23, 24, 25, 26, 27, 28, 30, 31]),
    ("2k", 2048, [8, 12, 13, 15, 16, 18, 26, 31]),
    ("1k", 1024, [16, 18, 26, 31]),
    ("512", 512, [26, 31]),
    ("256", 256, [31]),
]:
    idxs = [x for fold in folds for x in stratified_ids[fold]]
    assert len(set(idxs)) == target_size
    temp = make_train_subset(folds, name)
    assert (temp.groupby("split")[CINC_TARGETS].sum() > 0).all(axis=None)